In [1]:
from pathlib import Path
import sys

stage_dir = Path.cwd()
if not (stage_dir / "threshold_policy_utils.py").exists():
    stage_dir = Path.cwd() / "59. Política de threshold"
sys.path.insert(0, str(stage_dir.resolve()))

from threshold_policy_utils import (
    FOLDER_NAME,
    CONFIG_ID,
    PIPELINE_ID,
    MODEL_NAME,
    N_SPLITS,
    SEED,
    SW04_PARAMS,
    generate_sw04_oof_scores,
    results_dir,
)

In [2]:
LOG_TO_COMET = True

experiment_metadata = {
    "folder": FOLDER_NAME,
    "pipeline": PIPELINE_ID,
    "model": MODEL_NAME,
    "config_id": CONFIG_ID,
    "seed": SEED,
    "n_splits": N_SPLITS,
    "sample_weight_used": True,
    "weight_used_as_feature": False,
    "preprocessing_modified": False,
    "hyperparameters": SW04_PARAMS,
}
experiment_metadata

{'folder': '59. Política de threshold',
 'pipeline': 'P04',
 'model': 'XGBClassifier',
 'config_id': 'SW04',
 'seed': 42,
 'n_splits': 5,
 'sample_weight_used': True,
 'weight_used_as_feature': False,
 'preprocessing_modified': False,
 'hyperparameters': {'objective': 'binary:logistic',
  'eval_metric': 'logloss',
  'random_state': 42,
  'tree_method': 'hist',
  'n_jobs': -1,
  'missing': nan,
  'learning_rate': 0.03,
  'n_estimators': 800,
  'max_depth': 6,
  'min_child_weight': 1,
  'subsample': 0.9,
  'colsample_bytree': 0.9,
  'scale_pos_weight': 1.5346,
  'gamma': 0.0,
  'max_delta_step': 0,
  'reg_alpha': 0.5,
  'reg_lambda': 2.0}}

In [3]:
oof_df, fold_df = generate_sw04_oof_scores(log_to_comet=LOG_TO_COMET)

output_dir = results_dir()
print(f"OOF guardado en: {output_dir / 'p04_xgboost_sw04_oof_scores.csv'}")
print(f"Thresholds por fold guardados en: {output_dir / 'p04_xgboost_sw04_fold_thresholds.csv'}")
print(oof_df.shape)
display(oof_df.head())
display(fold_df[[
    "fold_id",
    "ams",
    "roc_auc",
    "threshold",
    "signal_percent",
    "precision",
    "recall",
    "f1",
    "signal_weight",
    "background_weight",
    "fit_seconds",
]])

/usr/lib/python3/dist-packages/requests/__init__.py:87: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (4.0.0) doesn't match a supported version!
  warnings.warn("urllib3 ({}) or chardet ({}) doesn't match a supported "
COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: xgboost, sklearn.
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/equipo-a/proyecto-1-taa/6dee94093c8d4e9a92095832fd447f2b

COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : 59. Política de t

OOF guardado en: /home/santig14/fing/taa/1_TallerAA/59. Política de threshold/results/p04_xgboost_sw04_oof_scores.csv
Thresholds por fold guardados en: /home/santig14/fing/taa/1_TallerAA/59. Política de threshold/results/p04_xgboost_sw04_fold_thresholds.csv
(250000, 5)


,event_id_valid,y_valid,weights_valid,scores_valid,fold_id
0,202596,0,0.307170,0.000847,1
1,175978,1,0.001503,0.051387,1
2,245641,0,2.292549,0.000087,1
3,130927,0,2.042184,0.013725,1
4,333946,0,2.446841,0.000037,1


,fold_id,ams,roc_auc,threshold,signal_percent,precision,recall,f1,signal_weight,background_weight,fit_seconds
0,1,1.715768,0.901329,0.053109,15.402,0.914037,0.410821,0.566861,47.741527,748.488462,14.161891
1,2,1.767991,0.897967,0.069019,13.194,0.928604,0.357535,0.516287,40.543732,502.538178,10.004812
2,3,1.566486,0.897142,0.036027,18.384,0.888816,0.476858,0.620703,57.456724,1316.313038,8.543805
3,4,1.624520,0.896885,0.061405,14.570,0.912835,0.388140,0.544680,44.003015,709.171025,7.979972
4,5,1.679953,0.898410,0.054097,15.548,0.915488,0.415397,0.571486,47.696848,780.349098,7.866514


In [4]:
fold_summary = {
    "ams_mean": fold_df["ams"].mean(),
    "ams_std": fold_df["ams"].std(ddof=1),
    "auc_mean": fold_df["roc_auc"].mean(),
    "auc_std": fold_df["roc_auc"].std(ddof=1),
    "threshold_mean": fold_df["threshold"].mean(),
    "threshold_median": fold_df["threshold"].median(),
    "threshold_std": fold_df["threshold"].std(ddof=1),
    "signal_percent_mean": fold_df["signal_percent"].mean(),
    "fit_seconds_mean": fold_df["fit_seconds"].mean(),
}
fold_summary

{'ams_mean': np.float64(1.6709436730898475),
 'ams_std': np.float64(0.07838221317771296),
 'auc_mean': np.float64(0.8983462164623113),
 'auc_std': np.float64(0.0017769280993700852),
 'threshold_mean': np.float64(0.05473148971796036),
 'threshold_median': np.float64(0.05409723520278931),
 'threshold_std': np.float64(0.012263396703869353),
 'signal_percent_mean': np.float64(15.419599999999999),
 'fit_seconds_mean': np.float64(9.711398722200101)}